# Persian homograph classifier (ParsBERT)

Given a sentence and a homograph word in it (e.g. `ترک`), the model picks the correct **reading**
(`tork` / `tark` / `tarak`). It is a context classifier: the target word is wrapped in marker tokens
`[H] ... [/H]`, and a linear head on top of ParsBERT chooses among *that word's own readings only*
(logits of other words' readings are masked out).

Pipeline: load CSV → strip diacritics → find and mark the target word → group-split by sentence →
fine-tune → evaluate (vs. a majority-class baseline) → save → reload and test.

**Expected CSV columns:** `Grapheme` (sentence), `Homograph Grapheme` (the word) and
`Homograph Phoneme` *or* `Mapped Homograph Phoneme` (the reading). Optional: `Source ID` / `Original_Index`
(used so that augmented copies of one sentence never end up in different splits) and `Source`.

In [1]:
!pip install -q torch transformers pandas numpy tqdm
import os, re, json, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

# ----------------------------- configuration -----------------------------
DATA_PATHS = ["PersianG2P.csv"]                    # one or more CSV files (they are concatenated)
MODEL_NAME = "HooshvareLab/bert-fa-zwnj-base"   # alternatives: "HooshvareLab/bert-base-parsbert-uncased", "HooshvareLab/bert-fa-base-uncased"
OUT_DIR    = "homograph_clf"                    # where the trained classifier is saved

MAX_LEN    = 128      # tokens per example (sentences are short; raise if yours are long)
BATCH_SIZE = 32       # lower to 16 if you run out of GPU memory
EPOCHS     = 4
LR         = 3e-5
WARMUP     = 0.06     # fraction of steps used for LR warm-up
VAL_FRAC   = 0.10     # fraction of *sentences* used for validation
TEST_FRAC  = 0.10     # fraction of *sentences* used for the final test
SEED       = 42
DROP_SINGLE_READING = True   # ignore words that only have one reading in the data (nothing to learn)
MAX_EXAMPLES = None          # e.g. 2000 for a quick debug run

MARK_OPEN, MARK_CLOSE = "[H]", "[/H]"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)
print("device:", DEVICE)

/home/karlsefni/projects/moss-finetuned/HomoRich-G2P-Persian/augmented_data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


## 1. Text helpers: normalisation and finding the homograph in the sentence

Diacritics are **stripped from the input** (some rows in the data are hand-diacritized, which would leak the
answer; at inference time the text will be plain anyway). The target token is found by matching the homograph
against each whitespace token, from strictest to loosest, so suffixed forms like `ترکی`, `پرهای`, `فکش`,
`می‌راند` and `بکن` are found too:

| score | rule | example (word `ترک`) |
|---|---|---|
| 0 | exact | `ترک` |
| 1 | exact after splitting on ZWNJ | `ترک‌خورده` |
| 2 | token starts with the word | `ترکی`, `ترکان` |
| 3 | a ZWNJ part starts with it | `می‌راند` for `راند` |
| 4 | token ends with it | `بکن` for `کن` |
| 5 | contained anywhere | |

In [2]:
ZWNJ = "‌"
DIAC_RE = re.compile("[ً-ٰٟـ]")      # harakat, hamza-above, superscript alef, tatweel
EDGE_RE = re.compile(r"^(\W*)(.*?)(\W*)$", re.S)           # leading / trailing punctuation around a token

def normalize(s):
    s = DIAC_RE.sub("", str(s))
    s = s.replace("ي", "ی").replace("ى", "ی").replace("ك", "ک").replace("_", ZWNJ)
    return re.sub(r"\s+", " ", s).strip()

# Mapped phoneme notation (a/ -> short a, etc.) back to the plain notation used in the "Homograph Phoneme" column
_MAPPED = {"/": "a", "a": "A", "$": "S", "c": "C", ";": "Z", "@": "?", "1": ""}
def mapped_to_plain(m):
    return "".join(_MAPPED.get(c, c) for c in str(m))

def split_token(tok):
    m = EDGE_RE.match(tok)
    return m.group(1), m.group(2), m.group(3)

def match_score(core, h):
    parts = core.split(ZWNJ)
    if core == h: return 0
    if h in parts: return 1
    if core.startswith(h): return 2
    if any(p.startswith(h) for p in parts): return 3
    if core.endswith(h): return 4
    if h in core: return 5
    return None

_KEY = str.maketrans({"آ": "ا", "أ": "ا", "إ": "ا", "ة": "ه"})
def match_key(s):                      # matching ignores آ/أ/إ vs ا (the data has اره written as آره)
    return s.translate(_KEY)

def find_target(tokens, h, max_score=5):
    best = None
    h = match_key(h)
    for i, t in enumerate(tokens):
        sc = match_score(match_key(split_token(t)[1]), h)
        if sc is not None and sc <= max_score and (best is None or sc < best[0]):
            best = (sc, i)
    return best                                   # (score, token index) or None

def mark_sentence(sentence, word, max_score=5):
    # returns (marked text, match score) or (None, None) if the word is not in the sentence
    toks = normalize(sentence).split(" ")
    hit = find_target(toks, normalize(word), max_score)
    if hit is None:
        return None, None
    sc, i = hit
    pre, core, post = split_token(toks[i])
    toks[i] = f"{pre} {MARK_OPEN} {core} {MARK_CLOSE} {post}"
    return re.sub(r"\s+", " ", " ".join(toks)).strip(), sc

print(mark_sentence("او تصمیم گرفت شهر را ترکِ کند.", "ترک"))
print(mark_sentence("دستش را می‌راند و رفت", "راند"))

('او تصمیم گرفت شهر را [H] ترک [/H] کند.', 0)
('دستش را [H] می\u200cراند [/H] و رفت', 1)


## 2. Load the data and mark the homographs

In [3]:
df = pd.concat([pd.read_csv(p, dtype=str, keep_default_na=False) for p in DATA_PATHS], ignore_index=True)
df = df.fillna("")
print("raw rows:", len(df), "| columns:", list(df.columns))

SENT_COL, WORD_COL = "Grapheme", "Homograph Grapheme"
assert SENT_COL in df and WORD_COL in df, "CSV needs 'Grapheme' and 'Homograph Grapheme' columns"

# reading label: plain notation if available, otherwise converted from the mapped notation
plain  = df["Homograph Phoneme"] if "Homograph Phoneme" in df else pd.Series([""] * len(df))
mapped = df["Mapped Homograph Phoneme"] if "Mapped Homograph Phoneme" in df else pd.Series([""] * len(df))
df["label"] = [p.strip() if p.strip() else mapped_to_plain(m).strip() for p, m in zip(plain, mapped)]
df["word"]  = df[WORD_COL].map(normalize)
df = df[(df["label"] != "") & (df["word"] != "")].reset_index(drop=True)

# group id = original sentence. Augmented copies (word-shuffled) share Original_Index / Source ID.
src = df["Source"] if "Source" in df else pd.Series(["all"] * len(df))
idc = next((c for c in ("Original_Index", "Source ID") if c in df and (df[c] != "").any()), None)
if idc:
    df["group"] = src.astype(str) + "|" + df[idc].astype(str)
else:  # fallback: bag-of-words key (identical for shuffled copies of a sentence)
    df["group"] = [" ".join(sorted(normalize(s).split(" "))) for s in df[SENT_COL]]
df["source"] = src.astype(str)
print("group column:", idc or "bag-of-words fallback", "| distinct sentences:", df["group"].nunique())

# mark the homograph in every sentence
marked, scores = zip(*[mark_sentence(s, w) for s, w in zip(df[SENT_COL], df["word"])])
df["text"], df["match_score"] = marked, scores
bad = df["text"].isna()
print(f"homograph not found in sentence: {bad.sum()} rows (dropped)")
if bad.any():
    print(df.loc[bad, [SENT_COL, WORD_COL]].head(5).to_string())
df = df[~bad].reset_index(drop=True)
print("match-score distribution:", df["match_score"].value_counts().sort_index().to_dict(),
      "(scores >= 4 are loose matches; skim a few below)")
print(df.loc[df["match_score"] >= 4, ["text", "word", "label"]].head(5).to_string())

df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

# keep only words with at least two readings
n_read = df.groupby("word")["label"].nunique()
if DROP_SINGLE_READING:
    single = n_read[n_read < 2].index
    if len(single): print("dropping single-reading words:", list(single)[:20])
    df = df[~df["word"].isin(single)].reset_index(drop=True)

if MAX_EXAMPLES and len(df) > MAX_EXAMPLES:
    df = df.sample(MAX_EXAMPLES, random_state=SEED).reset_index(drop=True)

print("\nexamples:", len(df), "| words:", df["word"].nunique(), "| readings:", df.groupby(["word", "label"]).ngroups)
print(df.groupby(["word", "label"]).size().rename("n").reset_index().sort_values(["word", "n"], ascending=[True, False]).head(30).to_string(index=False))

raw rows: 528934 | columns: ['Grapheme', 'Phoneme', 'Homograph Grapheme', 'Homograph Phoneme', 'Source', 'Source ID', 'Mapped Phoneme', 'Mapped Homograph Phoneme']
group column: Source ID | distinct sentences: 327484
homograph not found in sentence: 7806 rows (dropped)
                                                                         Grapheme Homograph Grapheme
286  هنگام پیاده‌روی روی پیاده‌روی‌خ‌زده، من روی سطح زمین خوردم و به زمین افتادم.                 سر
334                                 او به من قول داد تا نمره ریاضی‌اش را ۲۰ بگیرد                 قل
335                                       من به علی قول دادم تا هفته بعد پیشش روم                 قل
336                                  من به خودم قول دادم تا زندگی خوبی داشته باشم                 قل
337                                                    ما نباید زیر قول خود بزنیم                 قل
match-score distribution: {0.0: 250546, 1.0: 38214, 2.0: 21121, 3.0: 778, 4.0: 8278, 5.0: 741} (scores >= 4 are loose matche

## 3. Label space and the group-wise train / val / test split

Classes are `(word, reading)` pairs. For every word we remember which classes are possible; the model masks the rest.
The split is made **by sentence group**, so augmented copies of the same sentence cannot leak from train into test.

In [4]:
classes = sorted(set(zip(df["word"], df["label"])))
class2id = {c: i for i, c in enumerate(classes)}
words = sorted(df["word"].unique())
word2id = {w: i for i, w in enumerate(words)}
N_CLASSES, N_WORDS = len(classes), len(words)

word_mask = torch.zeros(N_WORDS, N_CLASSES, dtype=torch.bool)       # which classes each word may choose from
for (w, r), cid in class2id.items():
    word_mask[word2id[w], cid] = True

df["word_id"]  = df["word"].map(word2id)
df["class_id"] = [class2id[(w, r)] for w, r in zip(df["word"], df["label"])]

rng = np.random.RandomState(SEED)
groups = df["group"].unique().copy()
rng.shuffle(groups)
n_test = max(1, int(len(groups) * TEST_FRAC)); n_val = max(1, int(len(groups) * VAL_FRAC))
g_test, g_val, g_train = set(groups[:n_test]), set(groups[n_test:n_test + n_val]), set(groups[n_test + n_val:])
df["split"] = ["test" if g in g_test else "val" if g in g_val else "train" for g in df["group"]]
print(df["split"].value_counts().to_dict())

train_counts = df[df.split == "train"]["class_id"].value_counts()
missing = [classes[i] for i in range(N_CLASSES) if i not in train_counts.index]
if missing:
    print("WARNING: readings with no training example:", missing[:15])

/tmp/ipykernel_11450/3254626561.py:16: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(groups)


{'train': 250994, 'test': 31374, 'val': 31374}


## 4. Tokenizer, model, and datasets

Two new special tokens `[H]` / `[/H]` are added to the tokenizer and the embedding matrix is resized.
The classifier reads the hidden state at `[H]` and the mean of the word's sub-tokens, concatenates them and applies a linear layer.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"additional_special_tokens": [MARK_OPEN, MARK_CLOSE]})
OPEN_ID, CLOSE_ID = tokenizer.convert_tokens_to_ids([MARK_OPEN, MARK_CLOSE])

enc = tokenizer(df["text"].tolist(), truncation=True, max_length=MAX_LEN, add_special_tokens=True)
df["input_ids"] = enc["input_ids"]
ok = [(OPEN_ID in ids) and (CLOSE_ID in ids) for ids in df["input_ids"]]     # markers cut off by truncation?
print("examples whose marker was truncated (dropped):", len(ok) - sum(ok))
df = df[ok].reset_index(drop=True)
print("token length  mean/95%/max:", int(df.input_ids.map(len).mean()), int(df.input_ids.map(len).quantile(.95)), df.input_ids.map(len).max())
print(tokenizer.convert_ids_to_tokens(df["input_ids"][0]))

class HomographDS(Dataset):
    def __init__(self, frame):
        self.ids = frame["input_ids"].tolist(); self.w = frame["word_id"].tolist(); self.y = frame["class_id"].tolist()
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.ids[i], self.w[i], self.y[i]

def collate(batch):
    L = max(len(b[0]) for b in batch)
    ids = torch.full((len(batch), L), tokenizer.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (x, _, _) in enumerate(batch):
        ids[i, :len(x)] = torch.tensor(x); att[i, :len(x)] = 1
    return ids, att, torch.tensor([b[1] for b in batch]), torch.tensor([b[2] for b in batch])

def make_loader(split, shuffle):
    return DataLoader(HomographDS(df[df.split == split]), batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=collate)
train_dl, val_dl, test_dl = make_loader("train", True), make_loader("val", False), make_loader("test", False)

examples whose marker was truncated (dropped): 0
token length  mean/95%/max: 14 20 47
['[CLS]', '[H]', 'روی', '[/H]', 'دیوار', 'ننویس', '##ید', '.', '[SEP]']


In [6]:
class HomographClassifier(nn.Module):
    def __init__(self, encoder, word_mask, open_id, close_id, dropout=0.1):
        super().__init__()
        self.encoder = encoder
        H = encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(2 * H, word_mask.shape[1])
        self.register_buffer("word_mask", word_mask.clone())
        self.open_id, self.close_id = open_id, close_id

    def forward(self, input_ids, attention_mask, word_ids):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state     # B,T,H
        B, T, _ = h.shape
        pos = torch.arange(T, device=h.device)[None, :]
        start = (input_ids == self.open_id).float().argmax(1)                                      # position of [H]
        end_hit = (input_ids == self.close_id)
        end = torch.where(end_hit.any(1), end_hit.float().argmax(1), attention_mask.sum(1) - 1)    # position of [/H]
        span = (pos > start[:, None]) & (pos < end[:, None])                                       # tokens of the word
        span_vec = (h * span[..., None]).sum(1) / span.sum(1, keepdim=True).clamp(min=1)
        start_vec = h[torch.arange(B, device=h.device), start]
        logits = self.head(self.drop(torch.cat([start_vec, span_vec], dim=-1))).float()
        return logits.masked_fill(~self.word_mask[word_ids], -1e9)                                 # only this word's readings

encoder = AutoModel.from_pretrained(MODEL_NAME)
encoder.resize_token_embeddings(len(tokenizer))
model = HomographClassifier(encoder, word_mask, OPEN_ID, CLOSE_ID).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Loading weights: 100%|████████████████████| 197/197 [00:00<00:00, 14887.62it/s]
[transformers] BertModel LOAD REPORT from: HooshvareLab/bert-fa-zwnj-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

parameters: 119.3M


## 5. Train

AdamW + linear warm-up/decay, fp16 on GPU, gradient clipping. The model is evaluated on the validation set after every epoch and the best epoch is kept.

In [7]:
@torch.no_grad()
def predict_loader(model, loader):
    model.eval(); out = []
    for ids, att, w, y in loader:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            logits = model(ids.to(DEVICE), att.to(DEVICE), w.to(DEVICE))
        out.append(logits.argmax(-1).cpu())
    return torch.cat(out).numpy()

def accuracy(model, loader, frame):
    return float((predict_loader(model, loader) == frame["class_id"].values).mean())

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = EPOCHS * len(train_dl)
sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
loss_fn = nn.CrossEntropyLoss()
val_frame = df[df.split == "val"]

best_acc, best_state, history = -1.0, None, []
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); run = 0.0
    bar = tqdm(train_dl, desc=f"epoch {epoch}/{EPOCHS}", leave=False)
    for step, (ids, att, w, y) in enumerate(bar, 1):
        ids, att, w, y = ids.to(DEVICE), att.to(DEVICE), w.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            logits = model(ids, att, w)
        loss = loss_fn(logits, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
        if step % 50 == 0: bar.set_postfix(loss=f"{run / step:.4f}")
    val_acc = accuracy(model, val_dl, val_frame)
    history.append((epoch, run / len(train_dl), val_acc))
    print(f"epoch {epoch}: train loss {run / len(train_dl):.4f} | val acc {val_acc:.4f} | {time.time() - t0:.0f}s")
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print("best val acc:", round(best_acc, 4))

epoch 1: train loss 0.2817 | val acc 0.9344 | 1132s


epoch 2: train loss 0.1268 | val acc 0.9444 | 1185s


epoch 3: train loss 0.0778 | val acc 0.9480 | 1152s


epoch 4: train loss 0.0438 | val acc 0.9491 | 1167s
best val acc: 0.9491


## 6. Test

Compared against a **majority-class baseline**: for each word, always predict the reading that is most frequent in the training data.
If the model does not clearly beat that baseline on a word, it has not learned to use context for it.

In [8]:
test = df[df.split == "test"].copy()
train = df[df.split == "train"]
test["pred_id"] = predict_loader(model, test_dl)
test["pred"] = [classes[i][1] for i in test["pred_id"]]
test["correct"] = test["pred_id"] == test["class_id"]

majority = train.groupby("word")["label"].agg(lambda s: s.value_counts().index[0])
test["baseline_correct"] = test["word"].map(majority) == test["label"]

print(f"TEST sentences: {test.group.nunique()} | examples: {len(test)}")
print(f"model accuracy    : {test.correct.mean():.4f}")
print(f"majority baseline : {test.baseline_correct.mean():.4f}")
if test["source"].nunique() > 1:
    print("\nby source:\n", test.groupby("source")["correct"].agg(["size", "mean"]).round(4).to_string())

per_word = (test.groupby("word").agg(n=("correct", "size"), model=("correct", "mean"), baseline=("baseline_correct", "mean"))
            .round(3).sort_values("model"))
print("\nper word (worst first):\n", per_word.to_string())

for w in per_word.index[:5]:
    sub = test[test.word == w]
    if (~sub.correct).any():
        print(f"\nconfusion for {w}  (rows = true reading, columns = predicted):")
        print(pd.crosstab(sub["label"], sub["pred"]).to_string())

print("\nsome errors:")
print(test.loc[~test.correct, ["text", "label", "pred"]].head(10).to_string(index=False))

TEST sentences: 31374 | examples: 31374
model accuracy    : 0.9501
majority baseline : 0.4654

by source:
          size    mean
source               
GPT-4o  25275  0.9553
human    6099  0.9290

per word (worst first):
             n  model  baseline
word                          
قطعه       93  0.667     0.462
حل        100  0.700     0.500
فرانک      88  0.705     0.511
اسکی       99  0.707     0.515
تکه        96  0.750     0.500
مهم       122  0.762     0.451
پراند     102  0.765     0.480
ترساند     88  0.773     0.466
تاباند     95  0.779     0.400
کل        231  0.784     0.234
چپاند     116  0.784     0.500
چشم       140  0.786     0.293
کشتی      169  0.793     0.355
سپرد      106  0.802     0.396
جوشاند     94  0.809     0.489
گنگ       109  0.817     0.486
پژمرد      90  0.822     0.489
ادبا      107  0.822     0.458
شل        106  0.830     0.500
برگرداند  105  0.838     0.514
صرف       124  0.839     0.427
گرده      125  0.856     0.352
ترکاند    118  0.856     0.483
بین 

## 7. Save and reload

Saved to `OUT_DIR`: `model.pt` (weights), the tokenizer (with the marker tokens), the encoder config and `meta.json` (readings per word, settings).

In [9]:
def save_classifier(model, tokenizer, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    torch.save({k: v.cpu() for k, v in model.state_dict().items()}, f"{out_dir}/model.pt")
    model.encoder.config.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    meta = {"base_model": MODEL_NAME, "max_len": MAX_LEN, "mark_open": MARK_OPEN, "mark_close": MARK_CLOSE,
            "classes": [list(c) for c in classes], "words": words}
    with open(f"{out_dir}/meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1)

class Predictor:
    # wraps a trained classifier: predict the reading of a homograph in a sentence
    def __init__(self, model, tokenizer, meta, device):
        self.model, self.tok, self.device = model.to(device).eval(), tokenizer, device
        self.classes = [tuple(c) for c in meta["classes"]]
        self.words = meta["words"]; self.word2id = {w: i for i, w in enumerate(self.words)}
        self.max_len = meta["max_len"]
        self.readings = {}
        for i, (w, r) in enumerate(self.classes): self.readings.setdefault(w, []).append((i, r))

    @torch.no_grad()
    def predict(self, sentence, word, max_score=5):
        word = normalize(word)
        if word not in self.word2id: raise KeyError(f"'{word}' is not one of the trained homographs")
        text, sc = mark_sentence(sentence, word, max_score)
        if text is None: return None
        enc = self.tok(text, truncation=True, max_length=self.max_len, return_tensors="pt").to(self.device)
        wid = torch.tensor([self.word2id[word]], device=self.device)
        logits = self.model(enc["input_ids"], enc["attention_mask"], wid)[0]
        probs = torch.softmax(logits, -1).cpu()
        ranked = sorted(((r, float(probs[i])) for i, r in self.readings[word]), key=lambda x: -x[1])
        return {"word": word, "reading": ranked[0][0], "confidence": round(ranked[0][1], 3), "all": ranked, "match_score": sc}

    def predict_all(self, sentence, max_score=1):
        # naive: tries every trained homograph that appears (exact / ZWNJ-part match by default)
        out = []
        for w in self.words:
            r = self.predict(sentence, w, max_score)
            if r: out.append(r)
        return out

def load_classifier(out_dir, device=DEVICE):
    meta = json.load(open(f"{out_dir}/meta.json", encoding="utf-8"))
    tok = AutoTokenizer.from_pretrained(out_dir)
    enc_ = AutoModel.from_config(AutoConfig.from_pretrained(out_dir))
    o, c = tok.convert_tokens_to_ids([meta["mark_open"], meta["mark_close"]])
    m = HomographClassifier(enc_, torch.zeros(len(meta["words"]), len(meta["classes"]), dtype=torch.bool), o, c)
    m.load_state_dict(torch.load(f"{out_dir}/model.pt", map_location="cpu"))
    return Predictor(m, tok, meta, device)

save_classifier(model, tokenizer, OUT_DIR)
print("saved to", OUT_DIR, "->", sorted(os.listdir(OUT_DIR)))

saved to homograph_clf -> ['config.json', 'meta.json', 'model.pt', 'tokenizer.json', 'tokenizer_config.json']


## 8. Try it

First on held-out test sentences, then on your own sentences, then after reloading from disk to confirm the saved model gives identical answers.

In [10]:
pred = Predictor(model, tokenizer, json.load(open(f"{OUT_DIR}/meta.json", encoding="utf-8")), DEVICE)

print("--- random test sentences ---")
for _, r in test.sample(min(8, len(test)), random_state=1).iterrows():
    p = pred.predict(r[SENT_COL], r["word"])
    print(f"{'OK ' if p['reading'] == r['label'] else 'ERR'} {r['word']:6s} true={r['label']:8s} pred={p['reading']:8s} ({p['confidence']})  {normalize(r[SENT_COL])[:70]}")

print("\n--- your own sentences (edit freely) ---")
my_sentences = [
    ("او تصمیم گرفت شهر را ترک کند", "ترک"),
    ("زبان ترکی بسیار زیباست", "ترک"),
    ("لیوان روی میز افتاد و ترک خورد", "ترک"),
    ("برای رسیدن به آنجا باید دور میدان بچرخی", "دور"),
    ("او همیشه از بحث‌های بی‌فایده دور است", "دور"),
]
for s, w in my_sentences:
    if normalize(w) not in pred.word2id:
        print(f"(skipped: '{w}' is not in the trained word list)"); continue
    p = pred.predict(s, w)
    print(f"{w}: {p['reading']} ({p['confidence']})  all={p['all']}  |  {s}")

print("\n--- reload from disk and compare ---")
reloaded = load_classifier(OUT_DIR)
chk = test.sample(min(20, len(test)), random_state=2)
same = sum(pred.predict(r[SENT_COL], r["word"])["reading"] == reloaded.predict(r[SENT_COL], r["word"])["reading"] for _, r in chk.iterrows())
print(f"reloaded model agrees with in-memory model on {same}/{len(chk)} sentences")

--- random test sentences ---
OK  کمی    true=kammi    pred=kammi    (1.0)  کمی از انرژی‌ات را برای پروژه‌های آینده ذخیره کن.
OK  کشاند  true=keSAnad  pred=keSAnad  (0.964)  عادت قدیمی او را به کتابفروشی کشاند.
OK  کشاند  true=keSAnad  pred=keSAnad  (0.994)  او شلنگ را می کشاند
OK  جنب    true=janb     pred=janb     (1.0)  جنب مرکز خرید، یک کافی‌شاپ دنج وجود دارد.
OK  لنگ    true=leng     pred=leng     (0.997)  لنگ دستبندش از دستش افتاد و شکست.
OK  مد     true=madd     pred=madd     (0.999)  پدیده مد و جزر یکی از زیباترین مناظر طبیعی است.
OK  جست    true=jast     pred=jast     (1.0)  کبوتر به سوی آسمان آبی جست و پرواز کرد.
OK  چرخاند true=CarxAnd  pred=CarxAnd  (0.963)  وقتی که سرش را چرخاند، دوست قدیمی‌اش را دید.

--- your own sentences (edit freely) ---
ترک: tark (1.0)  all=[('tark', 0.9999923706054688), ('tarak', 5.879382570128655e-06), ('tork', 1.7751175391822471e-06)]  |  او تصمیم گرفت شهر را ترک کند
ترک: tork (1.0)  all=[('tork', 0.9999737739562988), ('tark', 2.3255663109011948e-